# Aula 5 — DataFrames com tidyverse
**Introdução à Programação para Pesquisa Biomédica · IBCCF/UFRJ**

> Runtime → Change runtime type → R

In [ ]:
library(tidyverse)

set.seed(42)
n <- 60
df <- tibble(
  gene      = paste0("GENE", sprintf("%03d", 1:n)),
  organismo = sample(c("Homo sapiens","Mus musculus","Danio rerio"), n, replace=TRUE),
  condicao  = sample(c("controle","tratamento"), n, replace=TRUE),
  expressao = round(rlnorm(n, 1.5, 0.8), 3),
  gc_pct    = round(runif(n, 35, 70), 1)
)
glimpse(df)

## 1. Inspecionar

In [ ]:
summary(df)
df |> count(organismo)

## 2. Selecionar e filtrar

In [ ]:
# Selecionar
df |> select(gene, expressao) |> head()

# Filtrar
humanos <- df |> filter(organismo == "Homo sapiens")
cat("Amostras humanas:", nrow(humanos), "\n")

# Múltiplas condições
df |> filter(expressao > 5, condicao == "tratamento") |> nrow()

## 3. Adicionar colunas (mutate)

In [ ]:
df <- df |>
  mutate(
    log2_expr = log2(expressao),
    grupo     = if_else(expressao > 5, "alto", "baixo")
  )
head(df)

## 4. Groupby e sumarização

In [ ]:
resumo <- df |>
  group_by(organismo) |>
  summarise(
    media_expr = mean(expressao),
    dp_expr    = sd(expressao),
    n_amostras = n(),
    .groups    = "drop"
  ) |>
  arrange(desc(media_expr))
print(resumo)

## 5. Join

In [ ]:
meta <- tibble(
  organismo  = c("Homo sapiens","Mus musculus","Danio rerio"),
  nome_comum = c("Humano","Camundongo","Zebrafish"),
  genoma_Mb  = c(3200, 2700, 1400)
)

df |> left_join(meta, by="organismo") |>
  select(gene, organismo, nome_comum, genoma_Mb, expressao) |> head()

## 6. Dados faltantes

In [ ]:
df_na <- df
df_na$expressao[sample(1:n, 8)] <- NA

cat("NAs por coluna:\n")
print(colSums(is.na(df_na)))

df_clean <- df_na |> drop_na(expressao)
cat("\nLinhas antes:", nrow(df_na), "| depois:", nrow(df_clean), "\n")